# TP1 — Offline Stage  
## Full-Order FEM Solutions and POD-Based Reduced Order Model

This notebook implements the **offline stage** of **Test Problem 1 (TP1)**.  
The objective of this stage is to construct a **Reduced Order Model (ROM)** based on **Proper Orthogonal Decomposition (POD)** starting from **high-fidelity Finite Element Method (FEM) solutions**.

The offline stage corresponds to the **Offline Module of the Inference Engine** described in the paper and provides all the reduced quantities required for efficient online simulations.

In [ ]:
import sys
from pyprojroot import here

PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

In [ ]:
from paths import INV_PATH, TP1_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MSH = PROJECT_ROOT

model_name = "rock"
xdmf_file_name = "rock.xdmf"

fig_dir = "figures/"
eig_fig = ABS_PATH + fig_dir + "sv.png"
error_fig = ABS_PATH + fig_dir + "error.png"
speed_up_fig = ABS_PATH + fig_dir + "speed_up.png"
samples_fig = ABS_PATH + fig_dir + "samples.png"

dbdir = ABS_PATH + "trainPOD/"

Import of packages

In [ ]:
import fenics as fe

from fem_problems.invTP1.finite_element import PoissonFEM
from fem_problems.invTP1.rbnics_pod import PODReduction

## Parametrized Poisson Problem

We consider the parametrized Poisson equation defined on the 3D rock domain $ \Omega \subset \mathbb{R}^3 $:


\begin{cases}
\Delta u(x,y,z;\boldsymbol{\mu}) = -(\alpha^2 + \beta^2)\pi^2 \lambda x \cos(\alpha \pi y)\sin(\beta \pi z), & \text{in } \Omega, \\
u(x,y,z;\boldsymbol{\mu}) = \lambda x \cos(\alpha \pi y)\sin(\beta \pi z), & \text{on } \partial \Omega,
\tag{1}
\end{cases}


where the parameter vector is defined as

\begin{equation}
\mu = (\lambda, \alpha, \beta). \tag{2}
\end{equation}

The availability of a manufactured analytical solution enables quantitative validation of the reduced-order approximation.

Load of the mesh elaborated in the notebook *00_ProblemSettings.ipynb*

In [ ]:
# Path to the XDMF file
file_path = ABS_PATH + xdmf_file_name

# Load mesh from file XDMF
mesh = fe.Mesh()
with fe.XDMFFile(file_path) as infile:
    infile.read(mesh)

## Parameters range

Here we fix the range of parameters. Specifically, $\mu = \left( \lambda, \alpha, \beta \right) \in \left[0, 1 \right]^3$

In [ ]:
mu_range = [
    (0, 1.),
    (0, 1.),
    (0, 1.),
]

In [ ]:
fem_p = PoissonFEM(mesh)
pod = PODReduction(mu_range, fem_p)

## Snapshot Generation

To construct the reduced-order model, we compute a set of **high-fidelity snapshots** by solving the FEM problem for different values of the parameter vector \( \boldsymbol{\mu} \).

Let

\begin{equation}
\mu_1, \mu_2, \dots, \mu_M
\subset \mathcal{P} \tag{3}
\end{equation}

be a set of sampled parameters.  
For each $ \mu_i $, a FEM solution $ u_h(\mu_i) $ is computed and stored.

The resulting snapshot matrix is defined as

\begin{equation}
S =
\begin{bmatrix}
u_h(\mu_1) & u_h(\mu_2) & \dots & u_h(\mu_M)
\end{bmatrix}. \tag{4}
\end{equation}

### Weak Formulation and FEM Discretization

Let $ V $ be a suitable Sobolev space.  
The weak formulation of the problem reads: find $ u \in V $ such that

\begin{equation}
a(u,v;\mu) = L(v;\mu) \quad \forall v \in V, \tag{5}
\end{equation}

with

\begin{equation}
a(u,v) = - \int_{\Omega} \nabla u \cdot \nabla v \, dx,
\qquad
L(v) = - \int_{\Omega} (\alpha^2 + \beta^2)\pi^2 \lambda x \cos(\alpha \pi y)\sin(\beta \pi z)\, v \, dx. \tag{6}
\end{equation}

The problem is discretized in space using the **Finite Element Method** with first-order Lagrange elements.

In [ ]:
num_training = 100
num_testing = 10

N_modes = num_training
tol = 1e-12

In [ ]:
training = pod.sampling_parameters(num_training)
testing = pod.sampling_parameters(num_testing, test=True)
pod.plot_samples(figsize = (6, 6), filename=samples_fig)

## Proper Orthogonal Decomposition (POD)

The reduced basis is constructed by applying **Proper Orthogonal Decomposition (POD)** to the snapshot matrix $ \mathbf{S} $.

The POD basis $ \{ \xi_1, \dots, \xi_k \} $ is obtained by solving the eigenvalue problem associated with the correlation matrix, retaining the modes that maximize the captured energy.

The reduced space is defined as

\begin{equation}
V_{\text{rb}} = \text{span}\{\xi_1, \dots, \xi_k\}, \tag{7}
\end{equation}

where the dimension $ k $ is selected according to an energy-based tolerance criterion.

In [ ]:
pod.pod_execution(N_modes, tol)

Plot of eigenvalues

In [ ]:
pod.plot_eigenvalues(filename=eig_fig)

Storing of reduced basis

In [ ]:
pod.store_reduction(directory=dbdir, filename=model_name)

Example of resolution for specifica values of parameters with ROM, FOM, and analytical solution

In [ ]:
mu_test = [.2, .6, .4]
mat, _ = pod.solve_rom(mu_test, pod.num_basis)
fom_m, _ = pod.fem_p.solve_fem(mu_test)
real_m, _ = pod.fem_p.exact_solution(mu_test)

print("ERROR FOM-EXACT")
e, f = pod.fem_p.compute_error(fom_m, real_m)
print("ERROR ROM-FOM")
a, b = pod.fem_p.compute_error(mat, fom_m)
print("ERROR ROM-EXACT")
c, d = pod.fem_p.compute_error(mat, real_m)

## Error Analysis

To assess the accuracy of the reduced-order model, we evaluate the approximation error with respect to the full-order FEM solution.

Given a test parameter $ \mu_{\mathrm{test}} $, the relative $ L^2 $-error is computed as

\begin{equation}
\varepsilon_{\text{rel}} =
\frac{\| u_h(\boldsymbol{\mu}_{\text{test}}) - u_{\text{rb}}(\boldsymbol{\mu}_{\text{test}}) \|_{L^2(\Omega)}}
{\| u_h(\boldsymbol{\mu}_{\text{test}}) \|_{L^2(\Omega)}}. \tag{8}
\end{equation}

In [ ]:
error, _ = pod.error_analysis()

Plot of error analysis results

In [ ]:
pod.plot_errors(error, filename=error_fig)

Plot of obtained speed up

In [ ]:
pod.plot_speed_up(error, filename=speed_up_fig)

Saving and showing table of obtained errors

In [ ]:
pod.save_table_error(error)

In [ ]:
pod.table_error(error)

## Offline Stage Outputs

At the end of the offline stage, the following quantities are available:

- POD basis functions and reduced operators,
- error indicators for reduced-order approximations.

These artifacts are stored and reused in the **online stage**, enabling fast and reliable simulations.